# 00 — Kiểm tra môi trường local

Chọn kernel **DDDCL Local** sau khi cài requirements. Notebook này kiểm tra môi trường và một bước cập nhật CPU; chưa chạy DDCL hay tải dữ liệu người dùng. Không nhập token vào notebook.


In [1]:
from pathlib import Path
import sys, platform
root = Path.cwd().resolve()
while not (root / "requirements.in").exists() and root != root.parent:
    root = root.parent
assert (root / "requirements.in").exists(), "Mở notebook từ checkout của dự án"
print("Python:", platform.python_version())
print("Interpreter:", sys.executable)
print("Project:", root)


Python: 3.11.0
Interpreter: D:\Academic_document\Term_5_year_3\ML\Project\LabEvaluate\ML_Project_DDDCL\.venv\Scripts\python.exe
Project: D:\Academic_document\Term_5_year_3\ML\Project\LabEvaluate\ML_Project_DDDCL


## Kiểm tra dependency

`pip check` chỉ xác nhận ràng buộc cài đặt, chưa chứng minh thuật toán đúng. Ghi môi trường thực tế khi báo cáo thí nghiệm.


In [2]:
import subprocess
import importlib.metadata as metadata
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
for package in ["numpy", "pandas", "scipy", "scikit-learn", "torch", "huggingface-hub", "wandb"]:
    print(package, metadata.version(package))


numpy 1.26.4
pandas 2.2.2
scipy 1.13.1
scikit-learn 1.5.1
torch 2.5.1
huggingface-hub 0.26.2
wandb 0.18.7


## Một bước cập nhật CPU

Ví dụ nhỏ kiểm tra forward/backward và seed; đây là fixture kỹ thuật, không phải kết quả benchmark.


In [3]:
import torch
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
torch.manual_seed(42)
model = torch.nn.Linear(2, 1)
x = torch.tensor([[0., 1.], [1., 0.], [1., 1.], [0., 0.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
before = model.weight.detach().clone()
optimizer.zero_grad()
loss = torch.nn.functional.binary_cross_entropy_with_logits(model(x), y)
loss.backward()
optimizer.step()
assert torch.isfinite(loss)
assert not torch.equal(before, model.weight.detach())
print("CPU update PASS; fixture loss =", float(loss.detach()))


CPU update PASS; fixture loss = 0.5965853333473206


## Bước tiếp theo

Đọc `docs/01_LOCAL_HF.md`, tạo/pin hosted data configuration. Agent sẽ tạo notebook audit data và notebook thuật toán ở các phase tiếp theo. Restart kernel + Run All trước khi đánh dấu môi trường pass.
